# GeoUNED STEP Conversion

Load one STEP file, convert it with GeoUNED, and view the generated cells in 3D.

In [ ]:
from pathlib import Path
import os
import shutil

freecad_bin = (Path(".venv") / "Library" / "bin").resolve()
os.environ["PATH_TO_FREECAD_LIBDIR"] = str(freecad_bin)
os.environ["PATH"] = f"{freecad_bin}{os.pathsep}{os.environ['PATH']}"

import geouned

examples = [Path("example_0.step"), Path("example_4.step")]
path = examples[0]
out = Path("outputs/notebook_run")
stem = path.stem

shutil.rmtree(out, ignore_errors=True)
out.mkdir(parents=True, exist_ok=True)

In [ ]:
geo = geouned.CadToCsg(
    settings=geouned.Settings(outPath=str(out), voidGen=False, debug=True)
)

geo.load_step_file(str(path), spline_surfaces="ignore")
geo.start()
geo.export_csg(
    geometryName=stem,
    outFormat=("openmc_py", "openmc_xml"),
    cellSummaryFile=True,
)

py_file = out / f"{stem}.py"
summary_file = out / f"{stem}_summary.txt"
cell_steps = sorted((out / "debug").glob("origSolid_*.stp"))

print(py_file)
print(f"{len(cell_steps)} cells")

In [ ]:
from IPython.display import Code, display

display(Code(filename=str(py_file), language="python"))

In [ ]:
import cadquery as cq
import plotly.graph_objects as go

colors = [
    "#2563eb", "#dc2626", "#16a34a", "#ea580c", "#7c3aed",
    "#0891b2", "#ca8a04", "#15803d", "#c026d3",
]

def mesh_from_step(path):
    shape = cq.importers.importStep(str(path)).val()
    vertices, triangles = shape.tessellate(0.08)
    return {
        "x": [v.x for v in vertices],
        "y": [v.y for v in vertices],
        "z": [v.z for v in vertices],
        "i": [t[0] for t in triangles],
        "j": [t[1] for t in triangles],
        "k": [t[2] for t in triangles],
    }

fig = go.Figure()
for n, step in enumerate(cell_steps, start=1):
    fig.add_trace(go.Mesh3d(
        **mesh_from_step(step),
        name=f"Cell {n}",
        color=colors[(n - 1) % len(colors)],
        flatshading=True,
        lighting={"ambient": 0.72, "diffuse": 0.65, "specular": 0.08, "roughness": 0.85},
    ))

fig.update_layout(
    template="plotly_white",
    width=900,
    height=650,
    margin={"l": 0, "r": 0, "t": 30, "b": 0},
    scene={
        "bgcolor": "#f8fafc",
        "aspectmode": "data",
        "xaxis": {"visible": False},
        "yaxis": {"visible": False},
        "zaxis": {"visible": False},
    },
)
fig.show(config={"displaylogo": False})